In [ ]:
!pip install langchain-openai

In [ ]:
from dotenv import load_dotenv
import os
#from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# # Check and print results
# doublecheck_env("example.env")
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
HF_API_KEY = userdata.get('HUGGINGFACE_API_KEY')
os.environ["HUGGINGFACE_API_KEY"] = HF_API_KEY
serper_api_key = userdata.get('SERPER_API_KEY')
os.environ["SERPER_API_KEY"] = serper_api_key
serp_api_key = userdata.get('SERPER_API_KEY')
os.environ["SERPER_API_KEY"] = serp_api_key

## Calculator example

In this example, the docstring and inferred arguments and argument types are used by the LLM to detetermine when and how to call the tool.

In [ ]:
from typing import Literal

from langchain.tools import tool


@tool
def real_number_calculator(
    a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]
) -> float:
    """Perform basic arithmetic operations on two real numbers."""
    print("🧮 Invoking calculator tool")
    # Perform the specified operation
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-4.1-nano",
    tools=[real_number_calculator],
    system_prompt="You are a helpful assistant",
)

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "what is 3.1125 * 4.1234"}]}
)
print(result["messages"][-1].content)

🧮 Invoking calculator tool
The result of multiplying 3.1125 by 4.1234 is approximately 12.8341.


In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "what is 3 * 4"}]})
print(result["messages"][-1].content)

🧮 Invoking calculator tool
The result of multiplying 3 by 4 is 12.


In [ ]:
### Example without Tool Support

In [ ]:
@tool("alt_calculator")
def my_alt_verse_calculator(a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]) -> float:
    """Perform basic arithmetic operations on two real numbers.
    """
    print("🧮 Invoking alt calculator tool")
    # Perform the specified operation
    if operation == "add":
        return a - b
    elif operation == "subtract":
        return a + b
    elif operation == "multiply":
      if b == 0:
            raise ValueError("Multiplication by zero is not allowed.")
      else:
        return a / b
    elif operation == "divide":
        return a * b
    else:
        raise ValueError(f"Invalid operation: {operation}")

In [ ]:
agent_without_tool_support = create_agent(
    model="openai:gpt-4.1-nano", #supposed to be a non-tool calling model, but still calls it.
    #model="openai:gpt-4", #supposed to be a non-tool calling model, but still calls it.
    #tools=[my_alt_verse_calculator],
    system_prompt="You are a helpful assistant",
)

In [ ]:
result = agent_without_tool_support.invoke(
    {"messages": [{"role": "user", "content": "what is 3.1125 * 4.1234"}]}
)
print(result["messages"][-1].content)

The product of 3.1125 and 4.1234 is approximately 12.8427.


## Adding a more detailed description
While a basic description is often sufficient, LangChain has support for enhanced descriptions. The example below uses one method: Google Style argument descriptions. Used with `parse_docstring=True`, this will parse and pass the arg descriptions to the model. You can rename the tool and change its description. This can be effective when you are sharing a standard tool but would like agent-specific instructions.

In [ ]:
from typing import Literal

from langchain.tools import tool


@tool(
    "calculator",
    parse_docstring=True,
    description=(
        "Perform basic arithmetic operations on two real numbers."
        "Use this whenever you have operations on any numbers, even if they are integers."
    ),
)
def real_number_calculator(
    a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]
) -> float:
    """Perform basic arithmetic operations on two real numbers.

    Args:
        a (float): The first number.
        b (float): The second number.
        operation (Literal["add", "subtract", "multiply", "divide"]):
            The arithmetic operation to perform.

            - `"add"`: Returns the sum of `a` and `b`.
            - `"subtract"`: Returns the result of `a - b`.
            - `"multiply"`: Returns the product of `a` and `b`.
            - `"divide"`: Returns the result of `a / b`. Raises an error if `b` is zero.

    Returns:
        float: The numerical result of the specified operation.

    Raises:
        ValueError: If an invalid operation is provided or division by zero is attempted.
    """
    print("🧮  Invoking calculator tool")
    # Perform the specified operation
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-4.1-nano",
    tools=[real_number_calculator],
    system_prompt="You are a helpful assistant",
)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "what is 3.0 * 4.0"}]})
print(result["messages"][-1].content)

🧮  Invoking calculator tool
The result of multiplying 3.0 by 4.0 is 12.0.


Let's check our [LangSmith Observability trace](https://smith.langchain.com/public/7d65902c-bd3c-4fc6-bbd3-7c1d66566fda/r) to see the tool description.

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "what is 3 * 4"}]})
print(result["messages"][-1].content)

🧮  Invoking calculator tool
The result of 3 multiplied by 4 is 12.


In [ ]:
pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 11.3 MB/s eta 0:00:00


In [ ]:
# Invoke DuckDuckGoSearchRun directly and then via LangChain Agent

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Brown tumour is seen in?")

'The brown tumor is a bone lesion that arises in settings of excess osteoclast activity, such as hyperparathyroidism. They are a form of osteitis fibrosa cystica. Brown tumors have a slightly greater frequency in primary than in secondary hyperparathyroidism (3% versus 2%). However, secondary hyperparathyroidism is much more common than primary hyperparathyroidism, therefore most brown tumors that are seen are associated with secondary hyperparathyroidism. See full list on radiopaedia.org In chronic renal disease, continual and excessive urinary calcium excretion can lower serum calcium level and lead to a rise in parathyroid hormonesecretion. This results in mobilization of skeletal calcium through rapid osteoclastic turnover of bone to maintain normal serum calcium levels. In localized regions where bone loss is particularly rapid... See full list on radiopaedia.org Well-defined, purely lytic lesions that provoke little reactive bone. The cortex may be thinned or expanded, but will n

In [ ]:
from langchain.agents import create_agent
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

agent = create_agent(
    model="openai:gpt-4.1-nano",
    tools=[search],
    system_prompt="You are a helpful assistant",
)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "Brown tumour is seen in?"}]})
print(result["messages"][-1].content)

A brown tumour is a reactive bone lesion that occurs in the context of hyperparathyroidism, typically osteitis fibrosa cystica. It is characterized by the proliferation of fibrous tissue, woven bone, and hemorrhage, giving it a brownish color due to hemosiderin deposition. 

Would you like me to search for more detailed or recent information about brown tumours?


In [ ]:
pip install -qU  langchain-community langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


# Invoke GoogleSerperAPIWrapper

In [ ]:
from langchain_community.utilities import GoogleSerperAPIWrapper

search = GoogleSerperAPIWrapper()

search.run("What time is it in EST?")

'10:32\u202fAM'

In [ ]:
from langchain.agents import create_agent
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import Tool

search_utility = GoogleSerperAPIWrapper()

# Wrap the utility in a Tool object
search = Tool(
    name="Google_Search",
    description="A Google search tool for answering questions and getting up-to-date information.",
    func=search_utility.run,
)

agent = create_agent(
    model="openai:gpt-4.1-nano",
    tools=[search],
    system_prompt="You are a helpful assistant. Please answer concise and brief."
)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "A young male patient presents with complete rectal prolapse. The surgery of choice is?"}]})
print(result["messages"][-1].content)

The surgery of choice for complete rectal prolapse is often rectopexy, with options including abdominal or perineal approaches. The most common procedures are:

- Abdominal rectopexy (laparoscopic or open)
- Perineal rectosigmoidectomy (Altemeier procedure)
- Perineal rectopexy (Delorme procedure) for selected cases

The specific choice depends on patient factors such as age and comorbidities.


# YouTube Search

In [ ]:
pip install -qU  youtube_search

In [56]:
from langchain_community.tools import YouTubeSearchTool

tool = YouTubeSearchTool()

# Return 5 results
tool.run("The deformity of tibia in triple deformity of the knee is?,5")

"['https://www.youtube.com/shorts/Er-QSAFSkIc', 'https://www.youtube.com/shorts/Bct-KGtBZ90', 'https://www.youtube.com/shorts/6CjCFbwm5f4', 'https://www.youtube.com/watch?v=8UkGrEkb6Zw&pp=ygU6VGhlIGRlZm9ybWl0eSBvZiB0aWJpYSBpbiB0cmlwbGUgZGVmb3JtaXR5IG9mIHRoZSBrbmVlIGlzPw%3D%3D', 'https://www.youtube.com/watch?v=baIM3Vaezp0&pp=ygU6VGhlIGRlZm9ybWl0eSBvZiB0aWJpYSBpbiB0cmlwbGUgZGVmb3JtaXR5IG9mIHRoZSBrbmVlIGlzPw%3D%3D']"

# GoogleSerperAPI

In [ ]:
from langchain.agents import create_agent
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import Tool

search_utility = GoogleSerperAPIWrapper()

# Wrap the utility in a Tool object
search = Tool(
    name="Google_Search",
    description="A Google search tool for answering questions and getting up-to-date information.",
    func=search_utility.run,
)

agent = create_agent(
    model="openai:gpt-4.1-nano",
    tools=[search],
    system_prompt="You are a helpful assistant. Please answer concise and brief."
)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "How to import HfApiModel from smolagents version 1.24.0 or its current location in smolagents documentation?"}]})
print(result["messages"][-1].content)

HfApiModel in smolagents can typically be imported directly from the library as follows:

```python
from smolagents import HfApiModel
```

This is consistent across recent versions, including 1.24.0.


In [ ]:
pip install -qU  google-search-results langchain-community

# Google Scholar Tool

In [57]:
import os

from langchain_community.tools.google_scholar import GoogleScholarQueryRun
from langchain_community.utilities.google_scholar import GoogleScholarAPIWrapper

In [58]:
os.environ["SERP_API_KEY"] = serp_api_key
googlescholartool = GoogleScholarQueryRun(api_wrapper=GoogleScholarAPIWrapper())

# Invoke tool directly to see if it's working or not.
googlescholartool.run("Post transplant lymphoma is ?")

'No good Google Scholar Result was found'

In [ ]:
get_ipython().system('pip show smolagents')

Name: smolagents
Version: 1.24.0
Summary: 🤗 smolagents: a barebones library for agents. Agents write python code to call tools or orchestrate other agents.
Home-page: 
Author: 
Author-email: Aymeric Roucher <aymeric@hf.co>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, jinja2, pillow, python-dotenv, requests, rich
Required-by: 


SmolAgents

In [ ]:
get_ipython().system('ls -R /usr/local/lib/python3.12/dist-packages/smolagents')

/usr/local/lib/python3.12/dist-packages/smolagents:
agents.py		       monitoring.py
agent_types.py		       prompts
cli.py			       __pycache__
default_tools.py	       pyodide_deno_executor.bak.py
_function_type_hints_utils.py  remote_executors.py
gradio_ui.py		       tmp.py
__init__.py		       tools.py
local_python_executor.py       tool_validation.py
mcp_client.py		       utils.py
memory.py		       vision_web_browser.py
models.py

/usr/local/lib/python3.12/dist-packages/smolagents/prompts:
code_agent.yaml  structured_code_agent.yaml  toolcalling_agent.yaml

/usr/local/lib/python3.12/dist-packages/smolagents/__pycache__:
agents.cpython-312.pyc
agent_types.cpython-312.pyc
cli.cpython-312.pyc
default_tools.cpython-312.pyc
_function_type_hints_utils.cpython-312.pyc
gradio_ui.cpython-312.pyc
__init__.cpython-312.pyc
local_python_executor.cpython-312.pyc
mcp_client.cpython-312.pyc
memory.cpython-312.pyc
models.cpython-312.pyc
monitoring.cpython-312.pyc
pyodide_deno_executor.bak.cpython-312.

In [ ]:
get_ipython().system('cat /usr/local/lib/python3.12/dist-packages/smolagents/models.py')

# Copyright 2024 The HuggingFace Inc. team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
import json
import logging
import os
import re
import uuid
import warnings
from collections.abc import Generator
from copy import deepcopy
from dataclasses import asdict, dataclass
from enum import Enum
from threading import Thread
from typing import TYPE_CHECKING, Any

from .monitoring import TokenUsage
from .tools import Tool
from .utils import RateLimiter, Retrying, _is_package_avail

In [ ]:
get_ipython().system('pip install ddgs')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 25.4 MB/s eta 0:00:00


# SmolAgents with DuckDuckGoSearch with Python Code Interpreter

In [ ]:
from smolagents import (
    CodeAgent,
    DuckDuckGoSearchTool,
    PythonInterpreterTool,
    InferenceClientModel # Updated import
)
import os

# Initialize model and tools
model = InferenceClientModel(model_id='Qwen/Qwen3-Next-80B-A3B-Thinking', api_key=HF_API_KEY) # Updated model instantiation with existing HF_API_KEY variable

agent = CodeAgent(
    tools=[DuckDuckGoSearchTool(), PythonInterpreterTool()],
    model=model
)

# Run agent
result = agent.run(
    'Search for the latest Python release date, '
    'then calculate how many days ago it was. '
)

print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for the latest Python release date, then calculate how many days ago it was.                             │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  search_results = web_search("latest Python release date")                                                        
  print(search_results)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Status of Python versions](https://devguide.python.org/versions/)
Status of Python versions ¶ The main branch is currently the future Python 3.15, and is the only branch that 
accepts new features. The latest release for each Python version can be found on the download page.

[Latest Python Version (2025) - What's New in Python 3.14?](https://www.liquidweb.com/blog/latest-python-version/)
Python continues to evolve, bringing powerful new features, enhanced security, and performance improvements with 
every release . The latest major version, Python 3.14 was officially released on October 7, 2025. Let's explore the
key features of Python's current version, how to download and install it, and what this release means for 
developers.

[Python: All Releases, End of Life, Release Date - VersionLog](https://versionlog.com/python/)
List releases of Python , end of life, end of support, support status, release date of each version, LTS versions. 
Python is a popular programming langua...

[Python Versions — Python - from None to AI](https://python3.info/about/versions.html)
Python Versions Important Since Python 3.9: PEP 602 -- Annual Release Cycle for Python New Python release every 12 
months (1 year) 12 months (1 year) release cycle 18 months (1.5 year) of bugfix updates 42 months (3.5 year) of 
security updates Python Release Cycle Since Python 3.9: PEP 602 -- Annual Release Cycle for Python Python 3.9-3.12:
one and a half years of full support, followed by ...

[Python 3.14 Released and Other Python News for November 2025](https://realpython.com/python-news-november-2025/)
Python 3.14 is officially out, Python 3.15 begins, and Python 3.9 reaches end of life. Plus, Django 6.0 first beta 
released, new PEPs, and more Python news.

[Python Release Python 3.14.0 | Python.org](https://www.python.org/downloads/release/python-3140/)
A new command-line interface to inspect running Python processes using asynchronous tasks. The pdb module now 
supports remote attaching to a running Python process. For more details on the changes to Python 3.14, see What's 
new in Python 3.14. Build changes PEP 761: Python 3.14 and onwards no longer provides PGP signatures for release 
artifacts.

[Python Release Dates: A Comprehensive Guide - CodeRivers](https://coderivers.org/blog/python-release-date/)
Python has become one of the most popular programming languages in the world, renowned for its simplicity, 
versatility, and vast library ecosystem. Understanding Python release dates is crucial for developers, as it 
impacts everything from code compatibility to the availability of new features and security updates. This blog post
will explore the ins and outs of Python release dates , providing ...

[Python Version History List — Latest Releases & Future 
Scenario](https://tahamehmood.medium.com/python-version-history-list-latest-releases-future-scenario-8b97994c1db9)
Python is one of the most powerful and popular programming languages, known for its simplicity, versatility, and 
scalability. Since its debut in the early 1990s, Python has evolved dramatically, introducing new features and 
performance improvements with each release .

[Download Python | Python.org](https://www.python.org/downloads/)
Looking for a specific release ? Python releases by version number: Release version Release date Click for more 
Python 3.14.3 Feb. 3, 2026 Download Release notes Python 3.14.2 Dec. 5, 2025 Download Release notes Python 3.14.1 
Dec. 2, 2025 Download Release notes Python 3.14.0 Oct. 7, 2025 Download Release notes

[What's new in Python 3.14 — Python 3.14.3 documentation](https://docs.python.org/3/whatsnew/3.14.html)
Summary - Release highlights ¶ Python 3.14 is the latest stable release of the Python programming language, with a 
mix of changes to the language, the implementation, and the standard library. The biggest changes include template 
string literals, deferred evaluation of annotations, and support for subinterprete

[Step 1: Duration 24.48 seconds| Input tokens: 2,205 | Output tokens: 3,407]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(14)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 14

[Step 3: Duration 6.28 seconds| Input tokens: 9,179 | Output tokens: 12,271]

14


SmolAgents

In [ ]:
get_ipython().system('cat /usr/local/lib/python3.12/dist-packages/smolagents/tools.py')

#!/usr/bin/env python
# coding=utf-8

# Copyright 2024 The HuggingFace Inc. team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
from __future__ import annotations

import ast
import inspect
import json
import logging
import os
import sys
import tempfile
import textwrap
import types
import warnings
from abc import ABC, abstractmethod
from collections.abc import Callable
from contextlib import contextmanager
from functools import wraps
from pathlib import Path
from typing impo

In [ ]:
from smolagents import CodeAgent, InferenceClientModel
from smolagents.tools import Tool as SmolAgentTool # Import SmolAgentTool
from langchain_community.tools.google_scholar import GoogleScholarQueryRun
from langchain_community.utilities.google_scholar import GoogleScholarAPIWrapper
import os
from typing import Any

# Re-initialize GoogleScholarAPIWrapper (ensure SERP_API_KEY is set)
os.environ["SERP_API_KEY"] = serp_api_key
scholar_tool_langchain = GoogleScholarQueryRun(api_wrapper=GoogleScholarAPIWrapper())

# Define a custom tool that wraps the LangChain tool, making it compatible with smolagents
class GoogleScholarTool(SmolAgentTool):
    name: str = "google_scholar"
    description: str = "A Google Scholar search tool for academic research. Input should be a string query."
    inputs = {"query": {"type": "string", "description": "The search query."}}
    output_type: str = "string"

    def forward(self, query: str) -> Any: # Changed __call__ to forward
        return scholar_tool_langchain._run(query)

# Initialize the InferenceClientModel with a valid model_id and API key
model = InferenceClientModel(model_id='Qwen/Qwen3-Next-80B-A3B-Thinking', api_key=HF_API_KEY)

# Create a CodeAgent with the custom Google Scholar tool
scholar_agent = CodeAgent(
    tools=[GoogleScholarTool()], # Pass an instance of our custom tool
    model=model
)

print("CodeAgent with Google Scholar tool initialized.")

CodeAgent with Google Scholar tool initialized.


In [ ]:
academic_result = scholar_agent.run('Find recent papers on large language models and medical diagnostics.')
print(academic_result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find recent papers on large language models and medical diagnostics.                                            │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  search_results = google_scholar(query="large language models medical diagnostics recent")                        
  final_answer(search_results)                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: No good Google Scholar Result was found

[Step 1: Duration 9.11 seconds| Input tokens: 2,077 | Output tokens: 1,555]

No good Google Scholar Result was found


In [ ]:
academic_result = scholar_agent.run('Capsid of viral structure')
print(academic_result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Capsid of viral structure                                                                                       │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  search_results = google_scholar(query="capsid viral structure definition")                                       
  print(search_results)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
No good Google Scholar Result was found

Out: None

[Step 1: Duration 20.21 seconds| Input tokens: 2,071 | Output tokens: 4,145]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("protein shell")                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: protein shell

[Step 2: Duration 16.59 seconds| Input tokens: 4,298 | Output tokens: 7,467]

protein shell
